# Loan Approval Prediction — Portfolio ML Project

This notebook documents the transformation of a basic loan approval model into a professional, reproducible machine learning workflow.

Key goals:
- Understand the dataset thoroughly.
- Build robust preprocessing with no target leakage.
- Engineer financial features.
- Compare multiple classifiers.
- Save a production-ready pipeline.
- Generate honest validation metrics.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

train_df = pd.read_csv('LP_Train.csv')
train_df.head()

In [ ]:
train_df.info()
train_df.isna().sum()

In [ ]:
print('Target distribution:\n', train_df['Loan_Status'].value_counts(dropna=False))

plt.figure(figsize=(5, 4))
sns.countplot(data=train_df, x='Loan_Status')
plt.title('Loan Approval Distribution')
plt.show()

In [ ]:
num_cols = train_df.select_dtypes(include=['number']).columns.tolist()
cat_cols = train_df.select_dtypes(exclude=['number']).columns.tolist()
print('Numerical columns:', num_cols)
print('Categorical columns:', cat_cols)

In [ ]:
train_df['Dependents'] = train_df['Dependents'].fillna('0')
train_df['Gender'] = train_df['Gender'].fillna('Male')
train_df['Married'] = train_df['Married'].fillna('Yes')
train_df['Self_Employed'] = train_df['Self_Employed'].fillna('No')
train_df['LoanAmount'] = train_df['LoanAmount'].fillna(train_df['LoanAmount'].median())
train_df['Loan_Amount_Term'] = train_df['Loan_Amount_Term'].fillna(train_df['Loan_Amount_Term'].median())
train_df['Credit_History'] = train_df['Credit_History'].fillna(train_df['Credit_History'].mode()[0])

train_df.isna().sum()

In [ ]:
train_df['Total_Income'] = train_df['ApplicantIncome'] + train_df['CoapplicantIncome']
train_df['Loan_to_Income'] = train_df['LoanAmount'] / train_df['Total_Income'].replace(0, np.nan)
train_df['Loan_Term_Years'] = train_df['Loan_Amount_Term'] / 12
train_df.head()[['ApplicantIncome','CoapplicantIncome','Total_Income','LoanAmount','Loan_to_Income','Loan_Term_Years']]

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

cat_features = ['Gender','Married','Dependents','Education','Self_Employed','Property_Area']
num_features = ['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term','Credit_History']

X = train_df[cat_features + num_features]
y = train_df['Loan_Status']

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features)
])

model = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=4000, class_weight='balanced', random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
probs = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
preds = (probs >= 0.5).astype(int)

print('Accuracy:', round(accuracy_score(y.map({'Y': 1, 'N': 0}), preds), 4))
print('Precision:', round(precision_score(y.map({'Y': 1, 'N': 0}), preds), 4))
print('Recall:', round(recall_score(y.map({'Y': 1, 'N': 0}), preds), 4))
print('F1:', round(f1_score(y.map({'Y': 1, 'N': 0}), preds), 4))

## Project structure and outputs

The full project is organized into a reusable folder structure with training scripts, prediction scripts, a Streamlit app, and generated CSV outputs.

In [ ]:
# This notebook serves as the narrative and EDA companion to the production project files.
print('Updated notebook ready for the loan approval prediction project.')